In [ ]:
# NOT EXPRESSED NOT MENTIONNED

import muon as mu
import numpy as np
import pandas as pd
import anndata

DATA_PATH = '/home/lemanic-g04/'

mdata = mu.read(DATA_PATH + 'GSE194315_binarized_mu.h5mu')
protein = mdata.mod['protein']

X = protein.X.toarray() if hasattr(protein.X, 'toarray') else protein.X
X = X.astype(bool)
protein_names = protein.var_names.to_numpy()

def humanize(names):
    names = list(names)
    if len(names) == 0:
        return ""
    if len(names) == 1:
        return names[0]
    if len(names) == 2:
        return f"{names[0]} and {names[1]}"
    return ", ".join(names[:-1]) + f", and {names[-1]}"

def cell_to_text(row):
    expressed = protein_names[row]
    if len(expressed) == 0:
        return "This cell does not express any of the measured surface proteins."
    return f"This cell expresses the following surface proteins: {humanize(expressed)}."

texts = [cell_to_text(X[i]) for i in range(X.shape[0])] # appliquer a chaques cellules

# Preview
for i in range(3):
    print(f"--- {protein.obs_names[i]} ---")
    print(texts[i])
    print()

# Save
mdata.mod['protein'].obs['text'] = texts
mdata.obs['text'] = texts
anndata.settings.allow_write_nullable_strings = True
out_path = DATA_PATH + 'GSE194315_binarized_with_text_onlyexpressed_mu.h5mu'
mdata.write(out_path)
print(f"Saved to {out_path}")

pd.DataFrame({'cell_id': protein.obs_names, 'text': texts}).to_csv(
    'text_conversion/cell_texts_onlyexpressed.csv', index=False
)
print("Also saved text_conversion/cell_texts.csv")